# Quick Start Guide to crystALL

This quick start guide will walk you through the main functionalities of **crystALL**. If you are running this notebook through pre-configured Codespaces, simply choose the `base` Kernel on the upper right corner of this notebook and you are ready to go as several steps have already been completed for you. If you picked it up from GitHub on your own, make sure you: (1) installed it correctly according to [README](../README.md); and (2) register for a client string to access [MPDD](http://mpdd.phaseslab.com) (optional).

### Prototype Identification

We start by importing the `CrystALLSearch` API class, which unifies all functionalities of **crystALL**. 

In [12]:
from crystall.crystALL import CrystALLSearch

For each composition of interest, `crystALLSearch` allows us to **identify**, **categorize** and **classify** their prototype structures.

In this notebook, the `AlO2` composition is used for example.

*Note: **crystALL** embbeds pySIPFENN's descriptors to represent and categorize prototype structures and Machine Learning models from [pySIPFENN](https://github.com/PhasesResearchLab/pySIPFENN) and [Alignn](https://github.com/usnistgov/alignn) to classify each prototype by their formation energies. The availability of these models are checked by default during the initialization.*

In [ ]:
AlO2Search = CrystALLSearch("AlO2")

**crystALL** prototypes are generated out of a pool of initial structures, which can either be informed locally, provided a directory with a set of `POSCAR` files, or fetched directly from the MPDD database.

If connecting to MPDD, first we need to establish the connection using the `MongoClient` from `pymongo` and a private MPDD client string.

In this example, a demo `client_string` is used to connect to MPPD using `connect_MPDD`. 

In [ ]:
from pymongo import MongoClient
import os; client_string = os.getenv('MONGODB_URI')

MPDD_database = MongoClient(client_string)['MPDD']
AlO2Search.connect_MPDD(MPDD_database)

Depending on the user approach, prototype structures can be generated using the `generate_local` or `generate_MPDD` methods of `CrystALLSearch`. 

These methods annonymize and run unique elemental substitutions over the input structures and return the complete list of prototypes as `POSCAR` files saved in a new local directory.

In this tutorial, we use `generate_MPDD`, which conveniently searches > 4.4 million structures present in MPDD.

*Note: By default, **crystALL** limits the generation to 1,000 prototypes to prevent the inadvertent creation of files.*

In [ ]:
AlO2Search.generate_MPDD(limit=1000)

At this stage users are encouraged to investigate the prototype files generated under the `AL1O2-curated-prototypes` directory.

### Prototype Categorization and Classification

To address the larger number of prototypes potentially identified, **crystALL** features the ability to categorize each structure according to their physical properties leveraging on pySIPFENN's descriptors.

By design, the `run_pysipfenn` method is used to calculate pySIPFENN's descriptors and also the formation energy predictions for all prototypes identified.

In [ ]:
AlO2Search.run_pysipfenn()

**crystALL** also allows the calculation of other physical and thermochemical properties for further classification and analysis.

Currently, the following optional methods are provided:
- `get_alignn_properties` # multi-property predictions, including formation energy
- `get_mpdd_convexhull` # convex-hull stability

In this example, we run only `get_mpdd_convexhull`, as `get_alignn_properties` predictions may take longer to complete when there are too many prototypes.

In [17]:
AlO2Search.get_mpdd_convexhull()

**crystALL** categorizes prototypes using clustering techniques such as `KMeans` and `DBSCAN`. Additionally, it utilizes formation energy predictions (Objectives) for the stability classification of prototypes.

By default, all previously identified prototypes are included in the cluster analysis. Optionally, prototypes can be down selected based on their formation energy, either by specifying a maximum number of the most stable entries (`max_prototypes`) or by setting a formation energy cutoff (`cutoff_energy`).

In this example, we filter the 100 most stable prototypes using `filter_prototypes` and cluster them using `cluster_prototypes_KMeans`.

In [ ]:
AlO2Search.filter_prototypes(max_prototypes=100)
AlO2Search.cluster_prototypes_KMeans(n_clusters=16)

At this stage `Top Candidate` prototypes are assigned, representing the entries with lowest formation energies per Cluster calculated for each available Objective.

### Data Visualization and Data Export

**crystALL** allows visualization of the clustering process by reducing its dimensionality to 2D and 3D spaces for plotting.

To generate publication-ready plots, we use the `plot_2D` and `plot_3D` functions from `crystALLSearch`.

In [ ]:
AlO2Search.plot_2D()
AlO2Search.plot_3D()

Finally, all clustering and property predictions data can be exported to 'csv' format using the `export_table` function in `crystALLSearch`.

In this example, we export the data for `allPrototypes` and `topCandidates`.

In [ ]:
AlO2Search.export_table(content = ['allPrototypes', 'topCandidates'])

At this stage, users can investigate the `allPrototypes.csv` and `topCandidates.csv` files generated. You are now ready to go back and use **crystALL** to manipulate your own data.

In the second tutorial, [`02.MetalHydrides.ipynb`](02.MetalHydrides.ipynb), a real-case example is demonstrated using **crystALL**, showcasing its role in aiding the discovery of new materials structures.